# Phase 2 – Prétraitement des données

**Objectif** : Préparer le dataset `classification_dataset.csv` (construit en Phase 0 sans data leakage)  
pour l'entraînement des modèles de Machine Learning.

**Variable cible** : `ligne_fragile`  
- `1` = ligne à risque de sous-desserte (pays à faible fréquentation ET fort CO₂/passager)  
- `0` = ligne non fragile

**Note** : `passengers` et `co2_per_passenger` sont **exclues des features** car elles ont servi  
à construire la variable cible (éviter tout data leakage).

## 1. Chargement du dataset

In [45]:
import pandas as pd
import numpy as np

# Chargement du nouveau dataset (Phase 0 corrigée)
df = pd.read_csv("../../../../data/ml/classification_dataset.csv")

print(f"Shape : {df.shape}")
print(f"\nColonnes disponibles :")
print(df.columns.tolist())
df.head()

Shape : (15538, 19)

Colonnes disponibles :
['fact_id', 'route_id', 'night_train', 'country_id', 'year_id', 'operator_id', 'is_night', 'distance_km', 'duration_min', 'country_code', 'country_name', 'operator_name', 'duration_hours', 'avg_speed_kmh', 'long_night_route', 'distance_category', 'duration_category', 'night_bonus', 'ligne_fragile']


,fact_id,route_id,night_train,country_id,year_id,operator_id,is_night,distance_km,duration_min,country_code,country_name,operator_name,duration_hours,avg_speed_kmh,long_night_route,distance_category,duration_category,night_bonus,ligne_fragile
0,1,1.0,ÖBB NJ 468 + NJ 469,4,15,1,True,1522.372257,961.498268,BE,Belgium,ÖBB,16.024971,95.0,1,long,slow,1,1
1,2,2.0,ÖBB NJ 408 + NJ 409,9,15,1,True,668.170688,422.002540,DE,Germany,ÖBB,7.033376,95.0,0,medium,medium,1,1
2,3,3.0,HŽPP 1820 + 1821,19,15,2,True,687.502916,550.002333,HR,Croatia,HŽPP,9.166706,75.0,0,medium,slow,1,1
3,4,5.0,ÖBB NJ 470 + NJ 471,9,15,1,True,694.342563,438.532145,DE,Germany,ÖBB,7.308869,95.0,0,medium,medium,1,1
4,5,6.0,ČD EN 442 + EN 443,8,15,3,True,576.000000,406.588235,CZ,Czech Republic,ČD,6.776471,85.0,0,medium,medium,1,1


## 2. Vérification de la variable cible

In [46]:
# Vérification de la distribution de ligne_fragile
print("Distribution de ligne_fragile :")
print(df["ligne_fragile"].value_counts())
print()
print("Répartition en % :")
print(df["ligne_fragile"].value_counts(normalize=True).round(3) * 100)

Distribution de ligne_fragile :
ligne_fragile
0    10424
1     5114
Name: count, dtype: int64

Répartition en % :
ligne_fragile
0    67.1
1    32.9
Name: proportion, dtype: float64


## 3. Sélection des variables

### Variables exclues (ne jamais utiliser comme features) :
- `fact_id`, `route_id`, `night_train` : identifiants / texte libre sans valeur prédictive
- `country_id`, `operator_id`, `year_id` : clés techniques (remplacées par les noms)
- `country_code` : redondant avec `country_name`
- `passengers`, `co2_per_passenger` : **utilisées pour construire la cible** → leakage si incluses
- `ligne_fragile` : c'est la cible

In [47]:
# Définition de la cible
TARGET = "ligne_fragile"

# Features numériques (à standardiser)
NUMERIC_FEATURES = [
    "distance_km",
    "duration_min",
    "duration_hours",
    "avg_speed_kmh",
]

# Features binaires (passthrough — déjà en 0/1)
BINARY_FEATURES = [
    "is_night",       # True/False → sera converti en 0/1
    "night_bonus",    # déjà en 0/1
    "long_night_route",  # déjà en 0/1
]

# Features catégorielles (à encoder)
CATEGORICAL_FEATURES = [
    "country_name",
    "operator_name",
    "distance_category",
    "duration_category",
]

ALL_FEATURES = NUMERIC_FEATURES + BINARY_FEATURES + CATEGORICAL_FEATURES

print(f"Nombre total de features : {len(ALL_FEATURES)}")
print(f"Features retenues : {ALL_FEATURES}")

Nombre total de features : 11
Features retenues : ['distance_km', 'duration_min', 'duration_hours', 'avg_speed_kmh', 'is_night', 'night_bonus', 'long_night_route', 'country_name', 'operator_name', 'distance_category', 'duration_category']


## 4. Conversion de `is_night` en entier

In [48]:
# is_night peut être un booléen (True/False) → convertir en 0/1
df["is_night"] = df["is_night"].astype(int)

print("Valeurs uniques de is_night après conversion :")
print(df["is_night"].unique())

Valeurs uniques de is_night après conversion :
[1 0]


## 5. Vérification des valeurs manquantes sur les features retenues

In [49]:
missing = df[ALL_FEATURES + [TARGET]].isna().sum()
print("Valeurs manquantes par colonne :")
print(missing[missing > 0] if missing.sum() > 0 else "Aucune valeur manquante ✅")

Valeurs manquantes par colonne :
Aucune valeur manquante ✅


## 6. Séparation X / y

In [50]:
X = df[ALL_FEATURES].copy()
y = df[TARGET].copy()

print(f"X shape : {X.shape}")
print(f"y shape : {y.shape}")
print(f"\nRépartition de y :")
print(y.value_counts())

X shape : (15538, 11)
y shape : (15538,)

Répartition de y :
ligne_fragile
0    10424
1     5114
Name: count, dtype: int64


## 7. Split train / test stratifié (80/20)

In [51]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,      # conserve la répartition des classes dans chaque split
    random_state=42
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nRépartition train : {y_train.value_counts().to_dict()}")
print(f"Répartition test  : {y_test.value_counts().to_dict()}")

X_train : (12430, 11)
X_test  : (3108, 11)

Répartition train : {0: 8339, 1: 4091}
Répartition test  : {0: 2085, 1: 1023}


## 8. Pipeline de prétraitement

- **Numériques** : `StandardScaler` (centrage-réduction)
- **Binaires** : `passthrough` (déjà en 0/1, aucune transformation nécessaire)
- **Catégorielles** : `OneHotEncoder` avec `handle_unknown='ignore'` (robustesse aux nouvelles catégories)

In [52]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            NUMERIC_FEATURES
        ),
        (
            "bin",
            "passthrough",     # booléens/entiers 0-1 → pas de transformation
            BINARY_FEATURES
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            CATEGORICAL_FEATURES
        )
    ],
    remainder="drop"   # toute colonne non listée est ignorée
)

print("Pipeline de prétraitement défini ✅")

Pipeline de prétraitement défini ✅


## 9. Application du prétraitement

In [53]:
# fit sur train seulement → évite le data leakage entre train et test
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f"X_train après prétraitement : {X_train_processed.shape}")
print(f"X_test  après prétraitement : {X_test_processed.shape}")
print()

# Vérification qu'il n'y a pas de NaN résiduels
nan_train = np.isnan(X_train_processed).sum()
nan_test  = np.isnan(X_test_processed).sum()
print(f"NaN dans X_train_processed : {nan_train}")
print(f"NaN dans X_test_processed  : {nan_test}")

X_train après prétraitement : (12430, 109)
X_test  après prétraitement : (3108, 109)

NaN dans X_train_processed : 0
NaN dans X_test_processed  : 0


## 10. Sauvegarde du préprocesseur et des splits

In [54]:
import joblib

# Sauvegarde du préprocesseur (réutilisé par l'API et le script predict.py)
joblib.dump(preprocessor, "../../../../data/ml/preprocessor.joblib")
print("✅ Préprocesseur sauvegardé : data/ml/preprocessor.joblib")

# Sauvegarde des splits pour les phases suivantes
joblib.dump((X_train, X_test, y_train, y_test), "../../../../data/ml/splits.joblib")
print("✅ Splits sauvegardés : data/ml/splits.joblib")

# Sauvegarde des noms de features pour l'API
joblib.dump(ALL_FEATURES, "../../../../data/ml/feature_names.joblib")
print("✅ Noms de features sauvegardés : data/ml/feature_names.joblib")

✅ Préprocesseur sauvegardé : data/ml/preprocessor.joblib
✅ Splits sauvegardés : data/ml/splits.joblib
✅ Noms de features sauvegardés : data/ml/feature_names.joblib


## 11. Résumé de la phase

In [55]:
print("=" * 55)
print("RÉSUMÉ PHASE 2 — PRÉTRAITEMENT")
print("=" * 55)
print(f"Dataset source      : classification_dataset.csv")
print(f"Variable cible      : {TARGET}")
print(f"Observations totales: {len(df)}")
print(f"Features retenues   : {len(ALL_FEATURES)}")
print(f"  - Numériques      : {len(NUMERIC_FEATURES)}")
print(f"  - Binaires        : {len(BINARY_FEATURES)}")
print(f"  - Catégorielles   : {len(CATEGORICAL_FEATURES)}")
print(f"Taille X_train      : {X_train.shape}")
print(f"Taille X_test       : {X_test.shape}")
print(f"Dimensions après OHE: {X_train_processed.shape[1]} features")
print()
print("Variables exclues (anti-leakage) :")
print("  passengers, co2_per_passenger → utilisées pour construire ligne_fragile")
print("  fact_id, route_id, country_id, operator_id, year_id → identifiants techniques")
print()
print("Fichiers produits :")
print("  data/ml/preprocessor.joblib")
print("  data/ml/splits.joblib")
print("  data/ml/feature_names.joblib")

RÉSUMÉ PHASE 2 — PRÉTRAITEMENT
Dataset source      : classification_dataset.csv
Variable cible      : ligne_fragile
Observations totales: 15538
Features retenues   : 11
  - Numériques      : 4
  - Binaires        : 3
  - Catégorielles   : 4
Taille X_train      : (12430, 11)
Taille X_test       : (3108, 11)
Dimensions après OHE: 109 features

Variables exclues (anti-leakage) :
  passengers, co2_per_passenger → utilisées pour construire ligne_fragile
  fact_id, route_id, country_id, operator_id, year_id → identifiants techniques

Fichiers produits :
  data/ml/preprocessor.joblib
  data/ml/splits.joblib
  data/ml/feature_names.joblib
